### Notebook 范围

### 目的
计算 EP100 分波与 O3/U/FWD 的多 case 关系，并显式测试不同 source windows。

### 科学问题
0008-01 中 EP100 W1+W2 与 O3 RMSE 的关系，在其他 case 或其他窗口中是否稳定？

### 方法
EP100 = mean(-ep2)，100 hPa，40-80N，正值表示更强 upward wave activity；不是 divergence。窗口包括 primary、lead0-30，以及可用时的 Jan、Feb、Jan-Feb。

### 输出
outputs/figures/03_ep100 和 outputs/tables/03_ep100。

### 解读
若 all waves 与 W1+W2 高相关，且 W1+W2 与 O3_RMSE 相关，说明 planetary waves 是主要 spread 来源。

### 注意
非一月初始化 case 没有 Jan 数据；calendar windows 会按 case init month 判断可用性。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 计算窗口敏感性和相关图

### 目的
为每个 case 计算多个 EP100 source window，并绘制相关热图和 selected scatter。

### 科学问题
EPFlux 到底用哪个时间范围？primary 和 lead0-30 是否给出类似结论？Jan/Feb 是否只在有数据的 case 中解释？

### 方法
O3_RMSE 使用 init-May30 against same-year BWCN；U mean 使用同一 target window；FWD 为 U60N50 threshold。

### 输出
03_EP100_window_sensitivity_all_cases、03_wave_component_relevance_heatmap、03 selected scatter。

### 解读
窗口敏感性若只在某个窗口强，说明机制可能依赖 timing；W1+W2 dominance 若稳定，说明 all-wave 主要由 planetary waves 控制。

### 注意
BWCN reference 不用于这里的 EP100；这里是成员间 spread/skill 关系。


In [ ]:
def ep_windows_for_case(case):
    windows = {
        "primary": source_windows_for_case(case)["primary"],
        "lead0_30": source_windows_for_case(case)["lead0_30"],
    }
    for label, window in [("Jan", MONTH_WINDOWS["Jan"]), ("Feb", MONTH_WINDOWS["Feb"]), ("Jan-Feb", ((1, 1), (2, 28)) )]:
        if label == "Jan-Feb":
            ok = case_month_window_available(case, "Jan") or case_month_window_available(case, "Feb")
        else:
            ok = case_month_window_available(case, label)
        if ok:
            windows[label] = window
    return windows

def case_source_table_for_window(case, window, label):
    meta = parse_case_name(case)
    ep = compute_all_ep100(case, window)
    o3, _ = load_hindcast_o3(case)
    ref, _ = load_bwcn_reference_o3(meta["year"])
    rmse = compute_o3_rmse(o3, ref, *target_window_for_case(case))
    out = ep.merge(rmse, on="member", how="outer") if not ep.empty else rmse.copy()
    out["case"] = case; out["year"] = meta["year"]; out["init_month"] = meta["init_month"]
    out["config"] = meta["config"]; out["perturbation"] = meta["perturbation"]
    out["source_window_key"] = label; out["source_window"] = window_to_label(window)
    out["EP100_definition"] = "mean(-ep2), 100 hPa, 40-80N; positive=stronger upward wave activity; not divergence"
    return out

fig_dir = figure_dir("03_ep100")
tab_dir = table_dir("03_ep100")
inv = discover_hindcast_cases()
all_metrics = []
for case in inv.loc[inv["can_source_diagnose"], "case"]:
    for label, window in ep_windows_for_case(case).items():
        tbl = case_source_table_for_window(case, window, label)
        if not tbl.empty:
            all_metrics.append(tbl)
metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
metrics_csv = tab_dir / "03_EP100_case_member_metrics_by_window.csv"
metrics.to_csv(metrics_csv, index=False)
# Root primary table for synthesis compatibility.
primary = metrics[metrics["source_window_key"] == "primary"].copy() if not metrics.empty else pd.DataFrame()
primary.to_csv(TAB_DIR / "03_EP100_case_member_metrics.csv", index=False)

corr_rows = []
for (case, key), sub in metrics.groupby(["case", "source_window_key"]) if not metrics.empty else []:
    for target in ["O3_RMSE"]:
        corr_rows.append({"case": case, "source_window_key": key, "metric": "wave1_plus_wave2 vs O3_RMSE", **finite_corr(sub["EP100_wave1_plus_wave2"], sub[target])})
    corr_rows.append({"case": case, "source_window_key": key, "metric": "all_waves vs W1+W2", **finite_corr(sub["EP100_all_waves"], sub["EP100_wave1_plus_wave2"])})
corr = pd.DataFrame(corr_rows)
corr_csv = tab_dir / "03_EP100_window_sensitivity_all_cases.csv"
corr.to_csv(corr_csv, index=False)
# Root copies used by final synthesis.
corr[corr["source_window_key"] == "primary"].rename(columns={"metric": "relationship"}).to_csv(TAB_DIR / "03_EP100_O3_U_FWD_correlations_all_cases.csv", index=False)
corr[corr["source_window_key"] == "primary"].to_csv(TAB_DIR / "03_wave_component_relevance_all_cases.csv", index=False)

for metric, fname, title in [
    ("wave1_plus_wave2 vs O3_RMSE", "03_EP100_W1W2_vs_O3_RMSE_window_sensitivity", "R(EP100 W1+W2, O3 RMSE) by source window"),
    ("all_waves vs W1+W2", "03_all_waves_vs_W1W2_window_sensitivity", "R(EP100 all waves, W1+W2) by source window"),
]:
    sub = corr[corr["metric"] == metric]
    mat = sub.pivot(index="case", columns="source_window_key", values="R") if not sub.empty else pd.DataFrame()
    fig, ax = plt.subplots(figsize=(10.5, max(4.5, 0.32 * len(mat) + 2)), constrained_layout=True)
    if mat.empty:
        ax.axis("off"); ax.text(0.5, 0.5, "No EP100 correlations", ha="center")
    else:
        order = [c for c in ["primary", "lead0_30", "Jan", "Feb", "Jan-Feb"] if c in mat.columns]
        mat = mat[order]
        im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=35, ha="right")
        ax.set_yticks(range(mat.shape[0]), mat.index)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat.iloc[i, j]
                ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2f}", ha="center", va="center", fontsize=8)
        ax.set_title(title + "\nEP100=mean(-ep2), 100 hPa, 40-80N; not divergence")
        fig.colorbar(im, ax=ax, label="Pearson R")
    savefig(fig, fname, fig_dir=fig_dir, notebook="03_EP100_wave_component_multicase.ipynb", scientific_question="EP100-O3 关系是否依赖 source window？", variables_windows="EP100 mean(-ep2), 100 hPa, 40-80N; windows primary/lead0-30/Jan/Feb/Jan-Feb; O3_RMSE init-May30", interpretation="稳定的高相关说明窗口选择不改变机制；只在个别窗口强说明 timing 很关键。", caveat="非一月初始化 case 没有初始化前 Jan/Feb 信息。", csv_table=corr_csv)
    plt.close(fig)

selected = [c for c in ["0008-01", "0013-02", "0014-02", "0019-02"] if c in set(primary.get("case", []))]
fig, axes = plt.subplots(1, max(1, len(selected)), figsize=(4.8 * max(1, len(selected)), 4.3), squeeze=False, constrained_layout=True)
if not selected:
    axes[0,0].axis("off"); axes[0,0].text(0.5, 0.5, "No selected cases", ha="center")
else:
    for ax, case in zip(axes.ravel(), selected):
        sub = primary[primary["case"] == case].dropna(subset=["EP100_wave1_plus_wave2", "O3_RMSE"])
        ax.scatter(sub["EP100_wave1_plus_wave2"], sub["O3_RMSE"], s=55, edgecolor="black", color="tab:purple")
        c = finite_corr(sub["EP100_wave1_plus_wave2"], sub["O3_RMSE"])
        ax.set_title(f"{case}: R={c['R']:.2f}, p={c['p']:.3f}, n={c['n']}")
        ax.set_xlabel("EP100 W1+W2\nmean(-ep2), 100 hPa, 40-80N")
        ax.set_ylabel("O3 RMSE (DU), init-May30")
for ax in axes.ravel()[len(selected):]:
    ax.axis("off")
savefig(fig, "03_EP100_W1W2_vs_O3_RMSE_selected_cases", fig_dir=fig_dir, notebook="03_EP100_wave_component_multicase.ipynb", scientific_question="Selected cases 中 EP100 W1+W2 是否解释 O3 RMSE？", variables_windows="primary source window; EP100 mean(-ep2), 100 hPa, 40-80N; O3 RMSE init-May30", interpretation="斜率和 R 显示成员间 upward wave activity 与臭氧误差的关系。", caveat="later-initialized cases 是 post-initialization forcing，不是 long-lead precursor。", csv_table=metrics_csv)
plt.show()
write_figure_guide()
